# Binary Classification — XGBoost Pipeline
Predicting `Class` from `TRAIN.csv`, generating submission on `TEST.csv`

In [33]:
# imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)
from xgboost import XGBClassifier

In [34]:
# load data
train_df = pd.read_csv("TRAIN.csv")
test_df  = pd.read_csv("TEST.csv")

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

Train: (43776, 48)
Test:  (10944, 48)


In [35]:
# data cleaning - check for nulls, duplicates, class balance
print(f"Missing values: {train_df.isnull().sum().sum()}")
print(f"Duplicates: {train_df.duplicated().sum()}")
print(f"\nClass distribution:")
print(train_df["Class"].value_counts())
print(f"\nClass ratio: {train_df['Class'].value_counts(normalize=True).to_dict()}")

Missing values: 0
Duplicates: 738

Class distribution:
Class
0    26465
1    17311
Name: count, dtype: int64

Class ratio: {0: 0.6045550073099415, 1: 0.3954449926900585}


In [36]:
# separate features and target
X = train_df.drop(columns=["Class"])
y = train_df["Class"]

test_ids = test_df["ID"]
X_test = test_df.drop(columns=["ID"])

In [37]:
# feature engineering — adding row-level stats to help the model
def add_features(df):
    df = df.copy()
    cols = df.columns.tolist()
    
    df["feat_mean"]     = df[cols].mean(axis=1)
    df["feat_std"]      = df[cols].std(axis=1)
    df["feat_min"]      = df[cols].min(axis=1)
    df["feat_max"]      = df[cols].max(axis=1)
    df["feat_median"]   = df[cols].median(axis=1)
    df["feat_range"]    = df["feat_max"] - df["feat_min"]
    df["feat_skew"]     = df[cols].skew(axis=1)
    df["feat_kurtosis"] = df[cols].kurtosis(axis=1)
    df["feat_q25"]      = df[cols].quantile(0.25, axis=1)
    df["feat_q75"]      = df[cols].quantile(0.75, axis=1)
    df["feat_iqr"]      = df["feat_q75"] - df["feat_q25"]
    df["feat_above_1"]  = (df[cols] > 1).sum(axis=1)
    df["feat_below_0"]  = (df[cols] < 0).sum(axis=1)
    df["feat_abs_sum"]  = df[cols].abs().sum(axis=1)
    df["feat_energy"]   = (df[cols] ** 2).sum(axis=1)
    return df

X      = add_features(X)
X_test = add_features(X_test)
print(f"Features after engineering: {X.shape[1]}")

Features after engineering: 62


In [38]:
# scaling — using RobustScaler since it handles outliers better
scaler = RobustScaler()
X_scaled      = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [39]:
# XGBoost — tuned params for better performance
scale_pos = len(y[y==0]) / len(y[y==1])

xgb = XGBClassifier(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=1.5,
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    verbosity=0
)

# 5-fold cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = cross_val_predict(xgb, X_scaled, y, cv=cv, n_jobs=-1)
oof_proba = cross_val_predict(xgb, X_scaled, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

print("5-Fold CV done.")

5-Fold CV done.


In [40]:
# threshold tuning — default 0.5 might not be best for imbalanced data
best_thresh = 0.5
best_f1 = 0

for t in np.arange(0.30, 0.70, 0.01):
    preds = (oof_proba > t).astype(int)
    score = f1_score(y, preds)
    if score > best_f1:
        best_f1 = score
        best_thresh = t

print(f"Optimal threshold: {best_thresh:.2f}")
print(f"F1 at optimal threshold: {best_f1:.5f}")

Optimal threshold: 0.39
F1 at optimal threshold: 0.98556


In [41]:
# evaluation metrics using the tuned threshold
final_oof = (oof_proba > best_thresh).astype(int)

print(f"Accuracy  : {accuracy_score(y, final_oof):.5f}")
print(f"Precision : {precision_score(y, final_oof):.5f}")
print(f"Recall    : {recall_score(y, final_oof):.5f}")
print(f"F1 Score  : {f1_score(y, final_oof):.5f}")
print(f"ROC-AUC   : {roc_auc_score(y, oof_proba):.5f}")

print("\nClassification Report:")
print(classification_report(y, final_oof, target_names=["Class 0", "Class 1"]))

print("Confusion Matrix:")
cm = confusion_matrix(y, final_oof)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"])
display(cm_df)

Accuracy  : 0.98860
Precision : 0.98764
Recall    : 0.98348
F1 Score  : 0.98556
ROC-AUC   : 0.99917

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.99      0.99      0.99     26465
     Class 1       0.99      0.98      0.99     17311

    accuracy                           0.99     43776
   macro avg       0.99      0.99      0.99     43776
weighted avg       0.99      0.99      0.99     43776

Confusion Matrix:


,Pred 0,Pred 1
Actual 0,26252,213
Actual 1,286,17025


In [43]:
# train on full data and predict on test set
xgb.fit(X_scaled, y)

test_probs = xgb.predict_proba(X_test_scaled)[:, 1]
test_preds = (test_probs > best_thresh).astype(int)

sub = pd.DataFrame({
    "ID": test_ids,
    "CLASS": test_preds
})

sub.to_csv("FINAL.csv", index=False)
print("FINAL.csv created successfully")
print(f"Predictions: {sub['CLASS'].value_counts().to_dict()}")
display(sub.head())

FINAL.csv created successfully
Predictions: {0: 6642, 1: 4302}


,ID,CLASS
0,1,1
1,2,0
2,3,1
3,4,0
4,5,0
